# Оценка Mean Average Precision

MAP системы, использующей модель IlyaGusev/saiga_yandexgpt_8b, равен 0.6296875.

Ноутбук содержит код для нахождения этой оценки. В качестве тестового набора выступают рекомендации, полученные в результате диалога модели с gpt-oss-120b (имитация собеседника). Количество диалогов - 40.

Эмбеддинги для элементов хранилища RAG найдены с помощью BAAI/bge-m3.



## Libraries

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
!pip install langchain
!pip install -qU langchain-community faiss-cpu
!pip install -qU langchain-huggingface

!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.0/444.0 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 11.1 MB/s eta 0:00:00


In [3]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from huggingface_hub import InferenceClient, login
from google.colab import userdata
from random import choice

## Model

Prompt для основной модели

In [5]:
main_prompt = """
Твоя роль - это эксперт-консультант по подбору товаров для хобби.

Твоя задача - найти идеальные товары на основе истории диалога или уточнить предпочтения.

Входные данные:

История диалога: История взаимодействий.
Список товаров: Название, описание, ссылка.
Количество сообщений от меня: Число моих сообщений в истории диалога.

Выходные данные:
Если (Количество сообщений от меня < 3): Сгенерировать один релевантный и наводящий вопрос для уточнения хобби и предпочтений. Вопрос должен быть конкретным и направленным на получение полезной информации для подбора. Вопрос должен быть построен так, чтобы он был максимально полезен для определения стиля и предпочтений собеседника (например, для украшения подарка это может быть вопрос о том, в каком стиле должен быть украшен подарок, какие цвета предпочитает получатель и т.п.)

Иначе, если (Недостаточно информации для определения конкретных интересов ИЛИ Последнее сообщение от собеседника слишком общее и требует уточнения): Сгенерировать один релевантный и наводящий вопрос для уточнения хобби и предпочтений. Вопрос должен быть конкретным и направленным на получение полезной информации для подбора. Вопрос должен быть построен так, чтобы он был максимально полезен для определения стиля и предпочтений собеседника (например, для украшения подарка это может быть вопрос о том, в каком стиле должен быть украшен подарок, какие цвета предпочитает получатель и т.п.)

Иначе, если (Есть подходящие товары в списке):

Если несколько товаров, то выбрать 2 наиболее релевантных, используя конкретные названия, ссылки и описания:

Я думаю, вам могут подойти эти товары:
[Название товара 1]: [Описание товара 1] ([Ссылка на товар 1])
[Название товара 2]: [Описание товара 2] ([Ссылка на товар 2])
...
Если один товар, используя конкретные названия, ссылки и описания:

Мне кажется, вам отлично подойдет [Название товара]: [Описание товара] ([Ссылка на товар])

Если (Количество сообщений от меня > 5):

Выбрать наиболее подходящий товар в списке, используя конкретные названия, ссылки и описания:

Мне кажется, вам отлично подойдет [Название товара]: [Описание товара] ([Ссылка на товар])

Правила:

Действовать строго по алгоритму: Сначала проверить количество сообщений и достаточность информации. Затем искать подходящие товары.
Избегать лишних фраз и объяснений.
Быть конкретным: Использовать информацию из списка товаров и истории диалога.
Сохранять вежливый и полезный тон.
Быть максимально кратким.
Избегать слова ‘пользователь’.
При выводе товаров указывать конкретные названия, описания и ссылки.
При выводе товаров обязательно писать ссылку на этот товар.
При генерации вопроса учитывать, что последнее сообщение собеседника могло быть очень общим и требовать уточнения.
Важно: Сначала проверять количество сообщений. Если их меньше 3, независимо от содержания последнего сообщения, генерировать только вопросы.
Не генерировать ответ больше, чем на 100 слов.
Не задавай больше, чем 2 вопроса за раз.
Если количество сообщений от меня больше, чем 5, то независимо от содержания диалога выбирать товар из списка.
Выводи товары только из входного списка.
"""

Промпт для модели gpt-oss-120b (имитация пользователя)

In [24]:
categories = ['Принадлежности для творчества', 'Отделочные материалы',
       'Детские книги', 'Бумага и бумажная продукция',
       'Упаковка для цветов', 'Принадлежности для рисования',
       'Подарочная упаковка', 'Фурнитура для бижутерии',
       'Принадлежности для шитья', 'Школьный текстиль',
       'Товары для художников', 'Фурнитура для сумок',
       'Универсальная упаковка', 'Флористика', 'Ленты',
       'Офисные принадлежности', 'Зеркала', 'Картины по номерам',
       'Гравюры', 'Плетение', 'Изготовление украшений', 'Роспись', 'Вазы',
       'Декоративные элементы для одежды', 'Музыкальные инструменты',
       'Инструменты', 'Вышивка', 'Алмазная мозаика', 'Молнии, замки',
       'Аквариумистика', 'Рисование', 'Принадлежности для лепки',
       'Фрески', 'Аксессуары для волос', 'Создание игрушек и кукол',
       'Посуда для сервировки стола', 'Фурнитура для штор',
       'Принадлежности для кроя', 'Часы', 'Косметички и органайзеры',
       'Элементы интерьера', 'Хранение в доме',
       'Принадлежности для макраме и плетения', 'Вязание',
       'Принадлежности для вышивания', 'Развивающие и обучающие игрушки',
       'Психология и эзотерика', 'Шитьё', 'Поделки из дерева',
       'Мыловарение', 'Письменные принадлежности',
       'Ёмкости для растений и аксессуары',
       'Оформление и организация праздника', 'Обустройство сада',
       'Маникюр и педикюр', 'Наборы для творчества', 'Конструкторы',
       'Боксы и органайзеры для хранения', 'Поделки из бумаги',
       'Игрушки для развлечений', 'Фурнитура для одежды', 'Мозаики',
       'Папки, системы архивации', 'Аппликации',
       'Чертёжные принадлежности', 'Изготовление свечей',
       'Цветы, флористика, декор', 'Валяние', 'Лепка', 'Скрапбукинг',
       'Торговое оборудование', 'Внешние дополнительные элементы',
       'Школьные товары', 'Брелоки и подвески',
       'Поделки из эпоксидной смолы', 'Опыты',
       'Кондитерские принадлежности и инвентарь для выпечки',
       'Куклы и пупсы', 'Пищевая упаковка', 'Бакалея', 'Грунты',
       'Мягкие игрушки', 'Тактическое снаряжение и амуниция',
       'Хобби и развлечения', 'Интерьерные сувениры',
       'Фоторамки и фотоальбомы', 'Одноразовая посуда',
       'Отдых и развлечения', 'Пазлы', 'Новый год', 'Текстиль для кухни',
       'Средства индивидуальной защиты', 'Сундучок Пандоры', 'Ткани',
       'Талисманы и фэншуй', 'Строительство дома', 'Гадания, пожелания',
       'Товары для детской комнаты', 'Подарочные наборы',
       'Хранение продуктов', 'Сюжетно-ролевые наборы', 'Шкатулки',
       'Декупаж', 'Броши и значки', 'Копилки',
       'Товары для ухода за одеждой и обувью', 'Музыкальные игрушки',
       'Обувь', 'Декор для кухни']

In [27]:
def user_prompt():
    return f"""
    Ты покупатель в магазине для хобби и сейчас общаешься с консультантом магазина для хобби.
    Ты должен обратиться к консультанту по поводу какой-либо потребности, а затем отвечать на вопросы консультанта.
    Обращение не должно быть длинным.
    Тематика потребности - {choice(categories)}.
    Возможный пример потребности: "Я бы очень хотел научиться рисовать". Не используй пример, генерируй свой.
    Ты начинаешь диалог.
    Не генерируй слишком длинные сообщения.
    """

Подключение FAISS для RAG

In [8]:
def load_faiss_index(embeddings, path="faiss_index"):
    db = FAISS.load_local(
        path, embeddings, allow_dangerous_deserialization=True
    )
    return db

Модель, обёрнутая в интерфейс

In [9]:
class SaigaWrapper:
    """ Wrapper class for IlyaGusev/saiga_yandexgpt_8b """

    def __init__(self, system_prompt):
        self.system_prompt = system_prompt
        self.MODEL_NAME = "IlyaGusev/saiga_yandexgpt_8b"

        self.model = AutoModelForCausalLM.from_pretrained(
            self.MODEL_NAME,
            load_in_8bit=True,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
        self.model.eval()

        self.tokenizer = AutoTokenizer.from_pretrained(self.MODEL_NAME)
        self.generation_config = GenerationConfig.from_pretrained(self.MODEL_NAME)

    def get_response(self, dialog, rag_information):

        messages = dialog.copy()
        messages.append({"role": "system", "content": rag_information + "\n" + self.system_prompt})

        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        data = self.tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
        data = {k: v.to(self.model.device) for k, v in data.items()}
        data.pop("token_type_ids", None)
        output_ids = self.model.generate(**data, generation_config=self.generation_config, max_new_tokens=400)[0]
        output_ids = output_ids[len(data["input_ids"][0]):]
        output = self.tokenizer.decode(output_ids, skip_special_tokens=True).strip()

        torch.cuda.empty_cache()

        return output

## Dialogs generation

HF Inference для модели собеседника (gpt-oss-120b)

In [10]:
login(userdata.get('HF_TOKEN2'))

client = InferenceClient(
    provider="auto",
    api_key=userdata.get('HF_TOKEN2'),
)

In [11]:
embeddings_for_faiss = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={'device': "cpu"})
faiss_db = load_faiss_index(embeddings_for_faiss, "/../data/processed/faiss_index")

model = SaigaWrapper(main_prompt)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/276 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/18.0M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

Функция для получения информации их хранилища RAG по запросу

In [12]:
def get_rag_addition(messages, rag_items_history):
    rag_query = "\n".join(
        [message["content"] for message in filter(lambda x: x["role"] == "user", messages)]
    )
    rag_items_history.extend(
        [
            f"Название товара: {str(doc.metadata['name'])}. Ссылка на товар: {str(doc.metadata['url'])}" for doc in
            faiss_db.similarity_search(rag_query, k=5)
        ]
    )
    rag_items_history = rag_items_history[-10:]
    rag_info = "Выбранные товары:\n" + "\n".join(rag_items_history) + f"\nКоличество сообщений от тебя: {(len(messages) - 1) // 2}\n"

    return rag_items_history, rag_info

Генерация диалогов. Создаётся список, содержащий информацию о репликах и информации из RAG (товарах)

In [67]:
# dataset with dialogs, format:
# [[{"question": ..., "rag_content": ..., "answer": ...} for message in dialog] for dialog in dialogs]
dataset = []

count_of_dialogs = 0
count_of_dialogs_total = 40     # count of generated dialogs

while count_of_dialogs < count_of_dialogs_total:
    messages_for_user = [{"role": "system", "content": user_prompt()}]  # context for gpt
    messages_for_assistant = []                                         # context for current model
    current_rag = []                                                    # rag items

    data_per_dialog = []                                                # current dialog info

    torch.cuda.empty_cache()
    while True:
        user_mes = client.chat_completion(
            model="openai/gpt-oss-120b",
            messages=messages_for_user,
            temperature=0.8,
            max_tokens=1024
        ).choices[0].message.content
        # print(user_mes)

        messages_for_user.append({"role": "assistant", "content": user_mes})
        messages_for_assistant.append({"role": "user", "content": user_mes})

        current_rag, rag_addition = get_rag_addition(messages_for_assistant, current_rag)
        # print(rag_addition)
        answer = model.get_response(
            messages_for_assistant,
            rag_addition
        )
        # print(answer)

        messages_for_user.append({"role": "user", "content": answer})
        messages_for_assistant.append({"role": "assistant", "content": answer})

        data_per_dialog.append(
            {"question": user_mes, "rag_content": rag_addition, "answer": answer}
        )

        # save dialog only after recommendation
        if "http" in answer or "www" in answer:
            count_of_dialogs += 1
            dataset.append(data_per_dialog)

            # save every 5 dialogs
            if count_of_dialogs % 5 == 0:
                with open("/../data/raw/dialogs_for_saiga_base_40.json", "w", encoding="UTF-8") as o:
                    o.write(json.dumps(dataset))
            print(f"Generated dialogs: {count_of_dialogs}/{count_of_dialogs_total}.")
            break

        # dialog is valid only if model has answer less than after 6 additional questions
        if len(messages_for_assistant) > 14:
            break

with open("/../data/raw/dialogs_for_saiga_base_40.json", "w", encoding="UTF-8") as o:
    o.write(json.dumps(dataset))

Generated dialogs: 1/40.
Generated dialogs: 2/40.
Generated dialogs: 3/40.
Generated dialogs: 4/40.
Generated dialogs: 5/40.
Generated dialogs: 6/40.
Generated dialogs: 7/40.
Generated dialogs: 8/40.
Generated dialogs: 9/40.
Generated dialogs: 10/40.
Generated dialogs: 11/40.
Generated dialogs: 12/40.
Generated dialogs: 13/40.
Generated dialogs: 14/40.
Generated dialogs: 15/40.
Generated dialogs: 16/40.
Generated dialogs: 17/40.
Generated dialogs: 18/40.
Generated dialogs: 19/40.
Generated dialogs: 20/40.
Generated dialogs: 21/40.
Generated dialogs: 22/40.
Generated dialogs: 23/40.
Generated dialogs: 24/40.
Generated dialogs: 25/40.
Generated dialogs: 26/40.
Generated dialogs: 27/40.
Generated dialogs: 28/40.
Generated dialogs: 29/40.
Generated dialogs: 30/40.
Generated dialogs: 31/40.
Generated dialogs: 32/40.
Generated dialogs: 33/40.
Generated dialogs: 34/40.
Generated dialogs: 35/40.
Generated dialogs: 36/40.
Generated dialogs: 37/40.
Generated dialogs: 38/40.
Generated dialogs: 39

In [4]:
with open("/../data/raw/dialogs_for_saiga_base_40.json", "r", encoding="UTF-8") as f:
    dataset_json = json.load(f)

## Dialogs Evaluation

Оценка происходит с помощью Mean Average Precision по диалогам.

$MAP@K = \frac{1}{N}\sum^N_{j=1} ap@K_j$, где $K_j$ - количество рекомендаций в $j$-м диалоге, $N$ - количество диалогов;

$ap@K_j = \frac{1}{K_j} \sum^{K_j}_{i=1}r(objects_i)precision@i$;

$precision@i=\frac{\#\{z:\ \ r(objects_z)=1,z<=i\}}{i}$;

$r(objects_i)=1$, если $i$-й объект выдачи должен стоять на $i$-м месте с точки зрения правильно отранжированного ряда, иначе $0$.

In [10]:
# better view of dialogs
def print_dialog(dialog):
    for mes in dialog:
        print("USER:")
        print(mes["question"])
        print()
        print("BOT")
        print(mes["answer"])
        print()

# ap@K
# Average precision at K
def average_precision(scores):

    def precision(k):
        return sum(scores[:k+1]) / len(scores[:k+1])

    K = len(scores)
    val = 0
    for k in range(K):
        val += scores[k] * precision(k)
    return val / K

In [20]:
# scores of AP for MAP
evaluation_scores = []

Валидация диалогов

In [21]:
for dialog in dataset_json:
    print_dialog(dialog)
    scores = list(map(int, input().split()))
    evaluation_scores.append(
        average_precision(scores)
    )

USER:
Здравствуйте! Я хочу попробовать рисовать портреты, но пока не знаю, какие карандаши и бумагу лучше взять для начала.

BOT
Какие материалы вы предпочитаете использовать при рисовании портретов? Например, акварель, масло, карандаш или уголь? Это поможет мне предложить более точные рекомендации.

USER:
Мне больше по душе графитовый карандаш — я хочу работать в монохроме и учиться передавать тональные переходы. Однако я также открыта к эксперименту с углем, если он поможет лучше передать тени и текстуру. Сейчас я ищу набор карандашей и подходящую бумагу для этих целей.

BOT
Так как ваше предпочтение — монохромное рисование карандашом с акцентом на передачу тонов и теней, а также возможность использования угля для создания текстуры, я предлагаю следующие два варианта:

**Набор для рисования:**
Название товара: Карандаши 12 цветов в тубусе, шестигранные, пластиковые.  
Ссылка на товар: https://www.sima-land.ru/7333128/karandashi-12-cvetov-v-tubuse-shestigrannye-plastikovye/  
Этот наб

In [22]:
# Current MAP
sum(evaluation_scores) / len(evaluation_scores)  # MAP@K, K=1...4

0.6296875